# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")

## 10. Shell 기반 Light 실험 실행

아래 셀들은 `scripts/*.sh` 와 `run_*.py` runner를 호출해서 Light baseline 실험을 순서대로 실행합니다.

사용 방법:

1. 먼저 바로 아래 `실험 설정 셀`에서 `RUN_NAME`, 모델 크기, 학습률, epoch 수 같은 값을 바꿉니다.
2. baseline에서 출발해, **이번 run에서 baseline 대비 바꾼 항목은 한 가지만** 유지합니다.
3. 그다음 필요한 실행 셀을 **위에서 아래 순서대로** 실행합니다.
4. pretrain 또는 finetune을 끝낸 뒤에는 아래 `실험 로그 요약 셀`로 결과를 기록합니다.
5. 결과는 `artifacts/{RUN_NAME}/pretrain`, `artifacts/{RUN_NAME}/finetune`, `artifacts/{RUN_NAME}/test_eval` 아래에 저장됩니다.

주의:

- `RUN_NAME`을 바꾸지 않으면 이전 실험 결과를 같은 artifact 경로에 덮어쓸 수 있습니다.
- `PRETRAINED_CHECKPOINT`는 기본적으로 같은 `RUN_NAME`의 pretrain 결과를 가리키도록 설정되어 있습니다.
- `FINETUNED_CHECKPOINT`는 기본적으로 같은 `RUN_NAME`의 finetune 결과를 가리키도록 설정되어 있습니다.
- 긴 실험 전에 작은 값으로 smoke run을 먼저 해보는 것이 안전합니다.

In [ ]:
import os
from pprint import pprint

os.chdir(repo_dir)

# 이번 run에서는 baseline 대비 바꾼 항목을 한 가지만 유지하세요.
# 여러 값을 동시에 바꾸는 경우 발표용 비교 근거가 약해집니다.

# ------------------------------------------------------------------
# 1. 발표용 실험 메타데이터
# ------------------------------------------------------------------
RUN_NAME = "light_baseline"
STAGE = "pretrain"  # "pretrain" 또는 "finetune"
CHANGED_HYPERPARAM = ""
CHANGED_VALUE = ""
RUN_NOTE = ""
NEXT_ACTION = ""

# ------------------------------------------------------------------
# 2. 공통 실행 환경
# ------------------------------------------------------------------
PYTHON_BIN = "python"
SEED = 123
DEVICE = "auto"

# ------------------------------------------------------------------
# 3. 비교군 고정 정보
# ------------------------------------------------------------------
TRAIN_CHAR_LIMIT = 500000
VAL_CHAR_LIMIT = 0
TOKENIZER_CORPUS_CHARS = TRAIN_CHAR_LIMIT
VAL_SPLIT_RATIO = 0.08

# ------------------------------------------------------------------
# 4. Pretrain baseline 설정
# ------------------------------------------------------------------
VOCAB_SIZE = 2000
CONTEXT_LENGTH = 64
STRIDE = CONTEXT_LENGTH
EMB_DIM = 64
N_HEADS = 4
N_LAYERS = 1
DROP_RATE = 0.0
QKV_BIAS = False
BATCH_SIZE = 8
LEARNING_RATE = "3e-4"
WEIGHT_DECAY = "0.0"
NUM_EPOCHS = 5
EVAL_FREQ = 50
EVAL_ITER = 10
CKPT_FREQ = 200
START_CONTEXT = "이 영화는"

# ------------------------------------------------------------------
# 5. Finetune baseline 설정
# ------------------------------------------------------------------
FINETUNE_BATCH_SIZE = 16
FINETUNE_LEARNING_RATE = "5e-5"
FINETUNE_WEIGHT_DECAY = "0.0"
FINETUNE_NUM_EPOCHS = 2
FINETUNE_DROP_RATE = 0.1

# ------------------------------------------------------------------
# 6. Artifact / checkpoint 경로
# ------------------------------------------------------------------
VOCAB_PATH = f"data/nsmc_bpe_vocab_{VOCAB_SIZE}.json"
PRETRAIN_ARTIFACT_DIR = f"artifacts/{RUN_NAME}/pretrain"
FINETUNE_ARTIFACT_DIR = f"artifacts/{RUN_NAME}/finetune"
PRETRAINED_CHECKPOINT = f"{PRETRAIN_ARTIFACT_DIR}/checkpoints/best.pt"
FINETUNED_CHECKPOINT = f"{FINETUNE_ARTIFACT_DIR}/checkpoints/best.pt"
TEST_EVAL_ARTIFACT_DIR = f"artifacts/{RUN_NAME}/test_eval"
EXPERIMENT_LOG_PATH = "artifacts/experiment_log.csv"

# ------------------------------------------------------------------
# 7. 현재 설정 요약 출력
# ------------------------------------------------------------------
config_summary = {
    "experiment_meta": {
        "RUN_NAME": RUN_NAME,
        "STAGE": STAGE,
        "CHANGED_HYPERPARAM": CHANGED_HYPERPARAM,
        "CHANGED_VALUE": CHANGED_VALUE,
        "RUN_NOTE": RUN_NOTE,
        "NEXT_ACTION": NEXT_ACTION,
    },
    "environment": {
        "PYTHON_BIN": PYTHON_BIN,
        "SEED": SEED,
        "DEVICE": DEVICE,
    },
    "fixed_comparison_info": {
        "TRAIN_CHAR_LIMIT": TRAIN_CHAR_LIMIT,
        "VAL_CHAR_LIMIT": VAL_CHAR_LIMIT,
        "TOKENIZER_CORPUS_CHARS": TOKENIZER_CORPUS_CHARS,
        "VAL_SPLIT_RATIO": VAL_SPLIT_RATIO,
    },
    "pretrain_baseline": {
        "VOCAB_SIZE": VOCAB_SIZE,
        "CONTEXT_LENGTH": CONTEXT_LENGTH,
        "STRIDE": STRIDE,
        "EMB_DIM": EMB_DIM,
        "N_HEADS": N_HEADS,
        "N_LAYERS": N_LAYERS,
        "DROP_RATE": DROP_RATE,
        "QKV_BIAS": QKV_BIAS,
        "BATCH_SIZE": BATCH_SIZE,
        "LEARNING_RATE": LEARNING_RATE,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "NUM_EPOCHS": NUM_EPOCHS,
        "EVAL_FREQ": EVAL_FREQ,
        "EVAL_ITER": EVAL_ITER,
        "CKPT_FREQ": CKPT_FREQ,
        "START_CONTEXT": START_CONTEXT,
    },
    "finetune_baseline": {
        "FINETUNE_BATCH_SIZE": FINETUNE_BATCH_SIZE,
        "FINETUNE_LEARNING_RATE": FINETUNE_LEARNING_RATE,
        "FINETUNE_WEIGHT_DECAY": FINETUNE_WEIGHT_DECAY,
        "FINETUNE_NUM_EPOCHS": FINETUNE_NUM_EPOCHS,
        "FINETUNE_DROP_RATE": FINETUNE_DROP_RATE,
    },
    "paths": {
        "VOCAB_PATH": VOCAB_PATH,
        "PRETRAIN_ARTIFACT_DIR": PRETRAIN_ARTIFACT_DIR,
        "FINETUNE_ARTIFACT_DIR": FINETUNE_ARTIFACT_DIR,
        "TEST_EVAL_ARTIFACT_DIR": TEST_EVAL_ARTIFACT_DIR,
        "EXPERIMENT_LOG_PATH": EXPERIMENT_LOG_PATH,
        "PRETRAINED_CHECKPOINT": PRETRAINED_CHECKPOINT,
        "FINETUNED_CHECKPOINT": FINETUNED_CHECKPOINT,
    },
}

pprint(config_summary, sort_dicts=False)


### 10.1 데이터 준비 셸

`scripts/prepare_nsmc.sh` 는 NSMC 원본 데이터를 준비하고, 이후 실험에서 공통으로 쓰는 파생 파일을 만듭니다.

무엇을 하나요:

- `ratings_train.txt`, `ratings_test.txt` 다운로드 또는 재사용
- `data/nsmc_lm_train.txt`, `data/nsmc_lm_val.txt` 생성
- `data/nsmc_sentiment_train.jsonl`, `val`, `test` 생성

언제 쓰나요:

- 처음 실험 환경을 만들 때
- 데이터 파일이 없을 때
- 감성 분류용 JSONL을 다시 만들고 싶을 때

보통은 한 번만 실행하면 되고, 이후에는 vocab / pretrain / finetune만 다시 돌려도 됩니다.

In [ ]:
import os

os.chdir(repo_dir)
!PYTHON_BIN={PYTHON_BIN} bash scripts/prepare_nsmc.sh

### 10.2 vocab 생성 셸

`scripts/build_vocab_light.sh` 는 현재 설정 셀의 `TOKENIZER_CORPUS_CHARS`, `VOCAB_SIZE`, `VOCAB_PATH` 값을 사용해서 BPE vocabulary를 학습하고 저장합니다.

무엇을 하나요:

- `data/nsmc_lm_train.txt` 일부를 읽음
- `BPETokenizer(vocab_size=...)` 학습
- `data/nsmc_bpe_vocab_*.json` 저장

언제 다시 돌리나요:

- `VOCAB_SIZE`를 바꿨을 때
- tokenizer 학습에 쓸 corpus 크기(`TOKENIZER_CORPUS_CHARS`)를 바꿨을 때
- 다른 `RUN_NAME`과 무관하게 vocab 파일 자체를 새로 만들고 싶을 때

반대로 tokenizer 조건을 안 바꿨다면, 이 셀은 다시 돌리지 않아도 됩니다.

주의:

- tokenizer 조건이 바뀌면 token-level loss 직접 비교가 공정하지 않을 수 있습니다.
- `VOCAB_SIZE`를 바꾼 실험은 `val loss`만 보지 말고, 시퀀스 길이 변화와 downstream 결과도 함께 해석하세요.

In [ ]:
import os

os.chdir(repo_dir)
!PYTHON_BIN={PYTHON_BIN} TRAIN_CHAR_LIMIT={TOKENIZER_CORPUS_CHARS} VOCAB_SIZE={VOCAB_SIZE} OUTPUT_PATH={VOCAB_PATH} bash scripts/build_vocab_light.sh

### 10.3 사전학습 셸

`scripts/run_pretrain_light.sh` 는 저장된 vocab을 읽어 GPT 언어모델 사전학습을 수행합니다.

무엇을 하나요:

- train / val corpus를 토큰화
- `CONTEXT_LENGTH`, `STRIDE`, `EMB_DIM`, `N_HEADS`, `N_LAYERS`, `DROP_RATE`로 GPT와 데이터 샘플링 구성
- `BATCH_SIZE`, `LEARNING_RATE`, `NUM_EPOCHS`, `EVAL_FREQ` 등으로 학습 수행
- `metrics.jsonl`, `loss.png`, `samples.txt`, `timing.json`, `best.pt`, `last.pt` 저장

언제 다시 돌리나요:

- 모델 구조나 pretrain 하이퍼파라미터를 바꿨을 때
- 다른 `RUN_NAME`으로 baseline을 새로 기록하고 싶을 때
- finetune에 사용할 새 backbone checkpoint가 필요할 때

이 셀은 가장 시간이 오래 걸릴 수 있으므로, 값 변경 후에는 artifact 경로와 `RUN_NAME`을 꼭 확인하는 것이 좋습니다.

현재 baseline에서는 `STRIDE = CONTEXT_LENGTH`로 두고, sample overlap을 만들지 않는 설정을 기본값으로 사용합니다.

In [ ]:
import os

os.chdir(repo_dir)
!PYTHON_BIN={PYTHON_BIN} TRAIN_CHAR_LIMIT={TRAIN_CHAR_LIMIT} VAL_CHAR_LIMIT={VAL_CHAR_LIMIT} VOCAB_SIZE={VOCAB_SIZE} VOCAB_PATH={VOCAB_PATH} ARTIFACT_DIR={PRETRAIN_ARTIFACT_DIR} CONTEXT_LENGTH={CONTEXT_LENGTH} STRIDE={STRIDE} EMB_DIM={EMB_DIM} N_HEADS={N_HEADS} N_LAYERS={N_LAYERS} DROP_RATE={DROP_RATE} QKV_BIAS={str(QKV_BIAS).lower()} BATCH_SIZE={BATCH_SIZE} LEARNING_RATE={LEARNING_RATE} WEIGHT_DECAY={WEIGHT_DECAY} NUM_EPOCHS={NUM_EPOCHS} EVAL_FREQ={EVAL_FREQ} EVAL_ITER={EVAL_ITER} CKPT_FREQ={CKPT_FREQ} START_CONTEXT='{START_CONTEXT}' SEED={SEED} DEVICE={DEVICE} bash scripts/run_pretrain_light.sh

### 10.4 감성 분류 finetune 셸

`scripts/run_finetune_light.sh` 는 감성 분류용 train/val JSONL과 pretrain checkpoint를 읽어 classifier-only fine-tuning을 수행합니다.

무엇을 하나요:

- `data/nsmc_sentiment_train.jsonl`, `data/nsmc_sentiment_val.jsonl` 로더 생성
- `PRETRAINED_CHECKPOINT` 에서 backbone 가중치 로드
- classifier를 붙여 `FINETUNE_*` 설정값으로 학습
- `metrics.jsonl`, `loss.png`, `accuracy.png`, `timing.json`, `best.pt`, `last.pt` 저장

언제 다시 돌리나요:

- 분류 학습률, epoch, batch size, dropout을 바꿨을 때
- 다른 pretrain checkpoint를 비교하고 싶을 때
- 같은 pretrain 결과에 대해 여러 finetune 조건을 비교할 때

이 셀은 train/val만 사용하고, test 데이터는 읽지 않습니다. test 평가는 바로 아래 전용 셸에서만 수행합니다.

In [ ]:
import os

os.chdir(repo_dir)
!PYTHON_BIN={PYTHON_BIN} VOCAB_SIZE={VOCAB_SIZE} VOCAB_PATH={VOCAB_PATH} PRETRAINED_CHECKPOINT={PRETRAINED_CHECKPOINT} ARTIFACT_DIR={FINETUNE_ARTIFACT_DIR} CONTEXT_LENGTH={CONTEXT_LENGTH} EMB_DIM={EMB_DIM} N_HEADS={N_HEADS} N_LAYERS={N_LAYERS} DROP_RATE={FINETUNE_DROP_RATE} QKV_BIAS={str(QKV_BIAS).lower()} BATCH_SIZE={FINETUNE_BATCH_SIZE} LEARNING_RATE={FINETUNE_LEARNING_RATE} WEIGHT_DECAY={FINETUNE_WEIGHT_DECAY} NUM_EPOCHS={FINETUNE_NUM_EPOCHS} SEED={SEED} DEVICE={DEVICE} bash scripts/run_finetune_light.sh

### 10.5 test-only 평가 셸

`scripts/run_test_eval_light.sh` 는 fine-tuning이 끝난 뒤, 저장된 분류기 checkpoint로 test 데이터에 대해 **gradient 없이** 최종 성능만 평가합니다.

무엇을 하나요:

- `data/nsmc_sentiment_test.jsonl` 로더 생성
- `FINETUNED_CHECKPOINT` 에서 fine-tuned classifier 가중치 로드
- `model.eval()` / `torch.no_grad()` 상태로 test loss, accuracy 계산
- `test_metrics.json`, `timing.json` 저장

언제 돌리나요:

- 실험 중간이 아니라, 최종 모델을 정한 뒤 마지막에 한 번
- 서로 다른 `RUN_NAME` 또는 checkpoint의 최종 test 성능을 비교할 때

이 셀만이 test 데이터를 읽습니다.

In [ ]:
import os

os.chdir(repo_dir)
!PYTHON_BIN={PYTHON_BIN} VOCAB_SIZE={VOCAB_SIZE} VOCAB_PATH={VOCAB_PATH} FINETUNED_CHECKPOINT={FINETUNED_CHECKPOINT} ARTIFACT_DIR={TEST_EVAL_ARTIFACT_DIR} CONTEXT_LENGTH={CONTEXT_LENGTH} EMB_DIM={EMB_DIM} N_HEADS={N_HEADS} N_LAYERS={N_LAYERS} DROP_RATE={FINETUNE_DROP_RATE} QKV_BIAS={str(QKV_BIAS).lower()} BATCH_SIZE={FINETUNE_BATCH_SIZE} DEVICE={DEVICE} bash scripts/run_test_eval_light.sh

### 10.6 실험 로그 요약 셀

pretrain 또는 finetune을 실행한 뒤, 현재 설정 셀의 메타데이터와 artifact를 읽어 발표용 실험 표 한 줄을 `artifacts/experiment_log.csv`에 기록합니다.

사용 방법:

- `STAGE`를 `pretrain` 또는 `finetune`으로 맞춥니다.
- `CHANGED_HYPERPARAM`, `CHANGED_VALUE`, `RUN_NOTE`, `NEXT_ACTION`을 채웁니다.
- 그 다음 이 셀을 실행하면 같은 `run + stage` 조합은 overwrite 됩니다.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from experiment_log import (
    NUMERIC_FIELDS,
    summarize_finetune_run,
    summarize_pretrain_run,
    upsert_experiment_log,
)

os.chdir(repo_dir)

if STAGE == "pretrain":
    row = summarize_pretrain_run(
        run_name=RUN_NAME,
        artifact_dir=PRETRAIN_ARTIFACT_DIR,
        changed_hyperparam=CHANGED_HYPERPARAM,
        changed_value=CHANGED_VALUE,
        note=RUN_NOTE,
        next_action=NEXT_ACTION,
    )
elif STAGE == "finetune":
    row = summarize_finetune_run(
        run_name=RUN_NAME,
        artifact_dir=FINETUNE_ARTIFACT_DIR,
        changed_hyperparam=CHANGED_HYPERPARAM,
        changed_value=CHANGED_VALUE,
        note=RUN_NOTE,
        next_action=NEXT_ACTION,
    )
else:
    raise ValueError(f"STAGE는 'pretrain' 또는 'finetune' 이어야 합니다: {STAGE}")

log_path = Path(EXPERIMENT_LOG_PATH)
upsert_experiment_log(log_path, row)

row_df = pd.DataFrame([row])
for column in NUMERIC_FIELDS:
    if column in row_df.columns:
        row_df[column] = pd.to_numeric(row_df[column], errors="coerce")

print(f"updated log: {log_path}")
display(row_df)


### 10.7 누적 실험 표 보기 셀

현재까지 기록된 `experiment_log.csv`를 읽어 stage별로 정렬된 누적 실험 표를 보여줍니다. 발표용 표 초안으로 바로 사용할 수 있습니다.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from experiment_log import NUMERIC_FIELDS, load_experiment_log_rows

os.chdir(repo_dir)

log_path = Path(EXPERIMENT_LOG_PATH)
rows = load_experiment_log_rows(log_path)
if not rows:
    print(f"아직 기록된 실험 로그가 없습니다: {log_path}")
else:
    df = pd.DataFrame(rows)
    for column in NUMERIC_FIELDS:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    stage_order = {"pretrain": 0, "finetune": 1}
    df["stage_order"] = df["stage"].map(stage_order).fillna(99)
    df = df.sort_values(["stage_order", "run"]).drop(columns=["stage_order"]).reset_index(drop=True)
    display(df)
